# Étape 4 : contrôles des règles de gestion et rapprochements entre sources

Ce notebook applique aux données chargées les règles de gestion écrites à la main dans `config/gestion.yaml`. Chaque règle est une instance d'un type générique du catalogue, avec ses paramètres (table, colonnes). Les dates, heures et nombres sont lus selon leur nature dans le dictionnaire enregistré (`data/config/dictionnaire.yaml`) : une valeur vide ou illisible rend la ligne non évaluable, sans nouvel écart.

Un écart est une anomalie, reste à arbitrer par le métier (avec, quand c'est possible, une correction proposée), ou est une information sur une pratique normale. Rien n'est corrigé. Les valeurs des colonnes sensibles ne sont jamais affichées.

Chaque mois : mettre à jour `PERIODE_FIN`, exécuter d'abord les notebooks `02c_dictionnaire` et `03_controles_format_doublon`, puis celui-ci en entier.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
%%time
# CHARGEMENT

from frigeo.chargement import charger_tout
from frigeo.controles import charger_dictionnaire_enregistre
from frigeo.dictionnaire import charger_meta

DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())

tables = charger_tout(perimetre)
# Sans dictionnaire enregistré, le message renvoie au notebook 02c_dictionnaire.
dictionnaire = charger_dictionnaire_enregistre()
print(len(tables), "tables,", sum(len(c) for c in dictionnaire.values()), "colonnes au dictionnaire")
print("dictionnaire de la période", charger_meta()["periode_fin"], ", données jusqu'à", PERIODE_FIN)

In [ ]:
# TYPES DE REGLES

from frigeo.gestion import catalogue_types

# Types disponibles pour écrire config/gestion.yaml, avec leurs paramètres.
display(catalogue_types())

In [ ]:
%%time
# REGLES DE GESTION

from frigeo.controles import ISSUE_A_ARBITRER, ISSUE_ANOMALIE
from frigeo.doublons import charger_declarations_doublons
from frigeo.gestion import (
    ISSUE_INFORMATION,
    charger_regles_gestion,
    chemin_regles_gestion,
    controler_gestion,
)

# Règles données par le métier dans config/gestion.yaml (aucune si le fichier est absent).
regles = charger_regles_gestion()
print(len(regles), "règles de gestion lues dans", chemin_regles_gestion())
# La clé de chaque table, affichée avec les écarts, vient des déclarations de doublons.
declarations = charger_declarations_doublons()

ecarts_gestion, bilan_gestion = controler_gestion(tables, dictionnaire, regles, declarations)
print(len(ecarts_gestion), "écarts, dont",
      int((ecarts_gestion["issue"] == ISSUE_ANOMALIE).sum()), "anomalies,",
      int((ecarts_gestion["issue"] == ISSUE_A_ARBITRER).sum()), "à arbitrer et",
      int((ecarts_gestion["issue"] == ISSUE_INFORMATION).sum()), "informations")
print(int(ecarts_gestion["proposition"].notna().sum()), "corrections proposées")

# Avertissements sur la configuration (par exemple une année sans jours fériés déclarés) :
# à lire avant le bilan, ils peuvent rendre des lignes non évaluables.
avertis = bilan_gestion[bilan_gestion["avertissements"].notna()]
print(len(avertis), "règles avec un avertissement")
for regle in avertis.itertuples():
    print(f"  {regle.ref} ({regle.libelle}) : {regle.avertissements}")
display(bilan_gestion.drop(columns=["libelle", "avertissements"]))

In [ ]:
# DETAIL PAR CONSTAT

display(
    ecarts_gestion.groupby(["ref", "table", "constat", "issue"])
    .agg(lignes=("constat", "size"), propositions=("proposition", "count"))
)